<a href="https://colab.research.google.com/github/Thorfast191/Monocular-Metric-Depth-Estimation/blob/main/MMDE_V1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
!pip install kaggle

In [46]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"mdarafatislam","key":"a008fff692ebc3a370386b72eab44439"}'}

In [47]:
%cd /content/drive/MyDrive/Datasets/NYU\ Depth\ V2
!ls
!ls images | head
!ls depths | head

/content/drive/MyDrive/Datasets/NYU Depth V2
kaggle.json  __pycache__
ls: cannot access 'images': No such file or directory
ls: cannot access 'depths': No such file or directory


In [48]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [49]:
!kaggle datasets download -d soumikrakshit/nyu-depth-v2

Dataset URL: https://www.kaggle.com/datasets/soumikrakshit/nyu-depth-v2
License(s): unknown
100% 4.10G/4.10G [01:22<00:00, 53.4MB/s]



In [ ]:
!unzip nyu-depth-v2.zip

Streaming output truncated to the last 5000 lines.
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/57.png  
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/58.jpg  
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/58.png  
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/59.jpg  
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/59.png  
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/6.jpg  
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/6.png  
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/60.jpg  
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/60.png  
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/61.jpg  
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/61.png  
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/62.jpg  
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/62.png  
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/63.jpg  
  inflating: nyu_data/data/nyu2_train/bedroom_0004_out/

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class NYUDataset(Dataset):
    def __init__(self, root_dir):
        self.img_dir = os.path.join(root_dir, "images")
        self.depth_dir = os.path.join(root_dir, "depths")
        self.files = sorted(os.listdir(self.img_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        img = cv2.imread(os.path.join(self.img_dir, file))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (320, 240)) / 255.0

        depth = cv2.imread(os.path.join(self.depth_dir, file), cv2.IMREAD_UNCHANGED)
        depth = cv2.resize(depth, (320, 240)).astype(np.float32)

        # Adjust scaling if needed
        depth = depth / 1000.0

        img = torch.tensor(img).permute(2, 0, 1).float()
        depth = torch.tensor(depth).unsqueeze(0)

        return img, depth

In [ ]:
dataset = NYUDataset("/content/drive/MyDrive/Datasets/NYU Depth V2")

img, depth = dataset[0]

print("Image shape:", img.shape)
print("Depth shape:", depth.shape)
print("Depth max:", depth.max())

In [ ]:
import torch.nn as nn
import torchvision.models as models

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(pretrained=True)

        self.layer0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.layer1 = nn.Sequential(resnet.maxpool, resnet.layer1)
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        x0 = self.layer0(x)
        x1 = self.layer1(x0)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return x0, x1, x2, x3, x4


class UpBlock(nn.Module):
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Sequential(
            nn.Conv2d(in_c + skip_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class DepthModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()

        self.up4 = UpBlock(2048, 1024, 1024)
        self.up3 = UpBlock(1024, 512, 512)
        self.up2 = UpBlock(512, 256, 256)
        self.up1 = UpBlock(256, 64, 128)

        self.final = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 1, 1)
        )

    def forward(self, x):
        x0, x1, x2, x3, x4 = self.encoder(x)

        d4 = self.up4(x4, x3)
        d3 = self.up3(d4, x2)
        d2 = self.up2(d3, x1)
        d1 = self.up1(d2, x0)

        return self.final(d1)

In [ ]:
class SILogLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, pred, target):
        mask = target > 0

        pred = pred[mask]
        target = target[mask]

        pred = torch.clamp(pred, min=1e-3)

        log_diff = torch.log(pred) - torch.log(target + 1e-6)

        return torch.sqrt((log_diff**2).mean() - 0.85 * (log_diff.mean()**2))

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

dataset = NYUDataset("/content/drive/MyDrive/Datasets/NYU Depth V2")
loader = DataLoader(dataset, batch_size=4, shuffle=True)

model = DepthModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = SILogLoss()

In [ ]:
from tqdm import tqdm
import os

os.makedirs("/content/drive/MyDrive/checkpoints", exist_ok=True)

for epoch in range(5):
    model.train()
    total_loss = 0

    for img, depth in tqdm(loader):
        img, depth = img.to(device), depth.to(device)

        pred = model(img)

        pred = torch.nn.functional.interpolate(
            pred, size=depth.shape[2:], mode='bilinear'
        )

        loss = criterion(pred, depth)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} Loss: {total_loss/len(loader)}")

    torch.save(model.state_dict(),
               f"/content/drive/MyDrive/checkpoints/model_{epoch}.pth")